# FIFA 2026 Player Performance

*  DSC 530
*  Final
*  Saebastion Cole
*  Chosen Data Source: FIFA 2026 Player Performance

## Dataset Selection Reasoning
With the end of the 2026 FIFA World Cup this data set has been recently released. Metric tracking is a huge part of sports especially international soccer. I chose this dataset to create new insights about player performance, roles, and reinforce some assumptions based on performance of players in higher regarded leagues compared to others.

The following are some of the analysis being reviewed:
- Are players that play on a career team in the champions league more impactful that a player that is not?
- Does the national team's climate and climate where the game took place play a part in the outcome?
- Where there any noticable outliers of positional play? (Did any players from one position do alot of another positions role in a game?)
- Does age effect movement (distance travelled).

## Setup

### Imports

In [353]:
# Begin by adding all of the needed imports at the top
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random as random

### Read Data

In [354]:
# Read in the player stats data

player_stats = pd.read_csv("data/fifa_world_cup_2026_player_performance.csv")

player_stats.head()

,player_id,player_name,age,nationality,team,jersey_number,position,height_cm,weight_kg,preferred_foot,...,possession_impact,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,player_of_match_awards,tournament_rating
0,P00055,Rodri Fati,26,Spanish,Spain,3,Goalkeeper,195,75,Left,...,1.1,44.2,55.9,42.0,51.8,0,0,242,0,5.8
1,P00070,Ansu Le Normand,19,Spanish,Spain,18,Midfielder,178,75,Right,...,3.5,38.2,43.7,31.1,52.7,0,3,342,0,5.5
2,P00066,Gavi Ramos,18,Spanish,Spain,14,Midfielder,177,72,Left,...,15.3,99.0,99.0,83.4,54.8,1,1,245,0,8.4
3,P00073,Pedro Cubarsi,20,Spanish,Spain,21,Forward,182,74,Right,...,1.2,19.8,42.3,40.9,78.5,5,3,422,0,6.7
4,P00059,Alvaro Oyarzabal,23,Spanish,Spain,7,Defender,191,81,Left,...,6.2,44.1,33.5,60.0,56.6,0,0,440,0,5.7


## Dataset Selection

This dataset from everything I have seen is very accurate and very complete. From the samples I have searched it contains the data for all of the games for the 2026 World Cup and all of the complete data for them. I have reviewed and seen no reason to believe there is bad data in it.

In [355]:
player_stats.info()

<class 'pandas.DataFrame'>
RangeIndex: 54600 entries, 0 to 54599
Data columns (total 75 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   player_id                 54600 non-null  str    
 1   player_name               54600 non-null  str    
 2   age                       54600 non-null  int64  
 3   nationality               54600 non-null  str    
 4   team                      54600 non-null  str    
 5   jersey_number             54600 non-null  int64  
 6   position                  54600 non-null  str    
 7   height_cm                 54600 non-null  int64  
 8   weight_kg                 54600 non-null  int64  
 9   preferred_foot            54600 non-null  str    
 10  club_name                 54600 non-null  str    
 11  market_value_eur          54600 non-null  int64  
 12  match_id                  54600 non-null  str    
 13  match_date                54600 non-null  str    
 14  stadium          

## Data Preperation

### Handle Missing Data / Cleaning

Not much cleaning is required with this dataset since it already has been cleaned and seems complete, I reviewed and didn't see any duplicates or anything.

So since players wont be switching national teams we can assume if they play one game for one then they are playing for that team for all other games.

### Re-encode variables

Most values are already numeric and the ones that aren't can be inferred that they aren't meant to be like nationality and team name. I will only add a column for the date of game to be a data time object.

#### Convert Match Date to Object


In [356]:
# Convert the match_date column to a datetime object
player_stats["match_date_obj"] = pd.to_datetime(player_stats["match_date"], format="%Y-%m-%d")

# Sort the dataset
player_stats = player_stats.sort_values(by="match_date_obj")

player_stats["match_date_obj"].head()

6317    2026-06-11
23907   2026-06-11
23906   2026-06-11
23905   2026-06-11
23904   2026-06-11
Name: match_date_obj, dtype: datetime64[us]

#### Set the index
Index is player and match IDs combined since no player should have multiple records for a match. Doing this will help lookup speeds and overall processing of the dataset.

In [357]:
# Create an index for the pair to help with filtering speed
player_stats.set_index(["player_name", "match_id"], inplace=True)
player_stats.head()

,,player_id,age,nationality,team,jersey_number,position,height_cm,weight_kg,preferred_foot,club_name,...,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,player_of_match_awards,tournament_rating,match_date_obj
player_name,match_id,,,,,,,,,,,,,,,,,,,,,
Giovanni Weah,M00122,P00653,33,American,United States,3,Goalkeeper,189,77,Right,LAFC,...,46.5,47.5,39.3,72.1,0,0,118,0,0.0,2026-06-11
Selim Saiss,M00460,P01060,21,Moroccan,Morocco,20,Midfielder,186,82,Right,Al-Hilal Omdurman,...,96.3,60.3,99.0,43.5,0,0,88,0,7.7,2026-06-11
Nayef Saibari,M00460,P01055,24,Moroccan,Morocco,15,Midfielder,175,70,Right,ASEC Mimosas,...,51.9,44.8,50.4,50.7,0,0,150,0,5.7,2026-06-11
Noussair El Khannouss,M00460,P01050,23,Moroccan,Morocco,10,Defender,180,74,Left,Club Africain,...,38.2,44.4,63.7,71.3,0,0,257,0,5.5,2026-06-11
Achraf Amellah,M00460,P01053,24,Moroccan,Morocco,13,Midfielder,186,77,Left,JS Kabylie,...,46.6,40.8,68.3,62.7,0,3,354,0,5.8,2026-06-11


### Data Transformation / Remove Skewness

#### Remove players that did not participate
For this the data will be cleaned picking only the players who have more than 0 minutes in. The rationale for this is FIFA allows world cup teams to consist of 26 players; Meanwhile only 11 are ever playing at a given time. The 2026 World Cup only allowed for 3 substitutions in regular time so nearly half of all players would not participate or have any meaningful statistics that would drastically skew the outcomes.

In [358]:
# Get data for players who actually competed
player_stats_active = player_stats[player_stats["minutes_played"] > 0]

player_stats_active.head()

,,player_id,age,nationality,team,jersey_number,position,height_cm,weight_kg,preferred_foot,club_name,...,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,player_of_match_awards,tournament_rating,match_date_obj
player_name,match_id,,,,,,,,,,,,,,,,,,,,,
Selim Saiss,M00460,P01060,21,Moroccan,Morocco,20,Midfielder,186,82,Right,Al-Hilal Omdurman,...,96.3,60.3,99.0,43.5,0,0,88,0,7.7,2026-06-11
Nayef Saibari,M00460,P01055,24,Moroccan,Morocco,15,Midfielder,175,70,Right,ASEC Mimosas,...,51.9,44.8,50.4,50.7,0,0,150,0,5.7,2026-06-11
Noussair El Khannouss,M00460,P01050,23,Moroccan,Morocco,10,Defender,180,74,Left,Club Africain,...,38.2,44.4,63.7,71.3,0,0,257,0,5.5,2026-06-11
Achraf Amellah,M00460,P01053,24,Moroccan,Morocco,13,Midfielder,186,77,Left,JS Kabylie,...,46.6,40.8,68.3,62.7,0,3,354,0,5.8,2026-06-11
Romain Aboukhlal,M00460,P01054,18,Moroccan,Morocco,14,Midfielder,183,85,Right,TP Mazembe,...,67.0,99.0,83.4,71.0,2,1,456,0,6.0,2026-06-11


### Normalization

Normalization will be done later in modeling of the dataset for initial analysis it should be accurate. I will use methods like applying logorithmic scale to convert the columns necessary.

## Exploratory Data Analysis (EDA) 

For my Analysis on this data I will focus on a few main points based on what is being analyzed. One of the core ratings I will be using is the player_rating. This is a value from 0-10 that helps quantify the impact the player had in the game and their overall performance. Alongside that I will focus on values like their club teams, their career income, age, and position. With focusing on some of these I should be able to filter out other features like shots (a goalie is not making shots on goal) and other more meaningful application.

#### 1.	Describe the Variable Characteristics with Summary 

- player_name: The name of the player
- player_rating: The calculated score of the player's performance that game.
- position: The position the player plays on the field (Like Striker, Midfield, Defender, and Goalie)
- club_name: This is the name of the team the player plays professionally for (This is seperate than an interanational team).
- market_value_eur: The estimated value of the player in euros.
- city: The city in which the World Cup game took place.
- match_result: the outcome of the game.
- distance_covered_km: How far the player moved in the game.
 

#### 2.	Conduct Univariate Analysis 

For this I am going to be somewhat selective on what I pick for each approach since there are a ton of features in this data set.

##### 1.	Describe the Variable Characteristics with Summary Statistics  

Player Rating Mean with / without players without time

In [359]:
# Calculate the average player rating

player_stats["player_rating"].mean()

np.float64(3.634699633699634)

In [360]:
player_stats_active["player_rating"].mean()

np.float64(6.288567082831612)

The comparison shows how much the scores get skewed by leaving in the players with time. 

Player Rating Median with / without players without time

In [361]:
# Calculate the average player rating

player_stats["player_rating"].median()

np.float64(5.5)

In [362]:
# Calculate the average player rating

player_stats_active["player_rating"].mean()

np.float64(6.288567082831612)

Even with using an approach that should help handle the skewed values it still is showing how much it gets effected.

Distance Covered Mean/Median

Distance Covered Mean

In [363]:
player_stats_active["distance_covered_km"].mean()

np.float64(6.9145414791811906)

In [364]:
player_stats_active["distance_covered_km"].median()

np.float64(7.5)

I am calculating mean and median together as I believe they compliment each other closely to make sense of the data. By looking at the distance covered we can see a higher median than mean. This tells me that were are outliers that are bringing the average down. This would make sense because of substitutions where players don't get enough time to run as much as well as goal keepers. Each team has a position that is moving very little.

###### Mode

Most Common Player Rating

In [365]:
player_stats_active["player_rating"].mode()

0    6.3
Name: player_rating, dtype: float64

Most Common Club Team

In [366]:
player_stats_active["club_name"].mode()

0    Gor Mahia
Name: club_name, dtype: str

Most Common Age

In [367]:
player_stats_active["age"].mode()

0    24
Name: age, dtype: int64

Most Common Position

In [368]:
player_stats_active["team"].mode()

0    Qatar
Name: team, dtype: str

###### Player Rating Variance

In [369]:
# Measure how spread out the player ratings are
player_stats_active["player_rating"].var()

np.float64(0.5418460247642566)

This tells me that majority of this feature is pretty normalized. Most of the data is contained nearby itself.

###### Skewness

Player Rating Skewness

In [370]:
# Measure how symmetric the distribution is
player_stats_active["player_rating"].skew()

np.float64(0.14284556881610416)

Market Value Skewness

In [371]:
player_stats_active["market_value_eur"].skew()

np.float64(2.957217826947612)

The player rating skewness isn't very significant. Meanwhile the Market value skewness is very skewed to the right. This is to be expected because of two reasons. We are evaluating for each player for each game and not making it unique to a player so this will make those signals much stronger. Also for Players like Cristiano Ronaldo and Lionel Messi they are paid much more than an average player.

###### Player Rating Kurtosis

In [372]:
# Measure how heavy the tails are compared to a normal distribution
player_stats_active["player_rating"].kurt()

np.float64(0.01287081014658309)

Kurtosis being so small is reinforcing my thought about the ratings being a very normal dataset. It definitely has outliers but most players can be considered "average" by World Cup standards (obviously any player at this level is well above average).

##### b.	Create visualization of the key variables to examine distribution or patterns 